# 02: Model Definitions
其他 notebook 開頭都要先 `%run 02_model.ipynb`

- `AFS_DSN_V4` — Full (~414M, 論文原版)
- `AFS_DSN_Lite` — Lite (~80M, 新增)
- `NasalSegDataset`, `CombinedLoss`, `dice_coefficient`

In [1]:
import subprocess, sys
for pkg in ['torch','numpy','pynrrd','PyWavelets','scipy','scikit-image','tqdm','matplotlib','pandas']:
    subprocess.check_call([sys.executable,'-m','pip','install',pkg,'-q'])
print('✅ Packages ready')


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip


✅ Packages ready



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
import numpy as np
import nrrd
import pywt
from pathlib import Path

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

Device: cuda
GPU: NVIDIA A40
VRAM: 47.7 GB


In [3]:
# ---- MultiScaleWavelet3D (原版完全一致) ----
class MultiScaleWavelet3D(nn.Module):
    def __init__(self, wavelet_scales=['db1','db2','db4']):
        super().__init__()
        self.wavelet_scales = wavelet_scales
    def forward(self, x):
        B,C,D,H,W = x.shape
        device = x.device
        all_coeffs = []
        for wavelet_type in self.wavelet_scales:
            coeffs_list = []
            for b in range(B):
                for c in range(C):
                    vol = x[b,c].detach().cpu().numpy()
                    coeffs = pywt.dwtn(vol, wavelet_type, mode='periodization')
                    bands = [coeffs['aaa'],coeffs['aad'],coeffs['ada'],coeffs['add'],
                             coeffs['daa'],coeffs['dad'],coeffs['dda'],coeffs['ddd']]
                    coeffs_list.append(torch.from_numpy(np.stack(bands,axis=0)).float().to(device))
            wt_out = torch.stack(coeffs_list,dim=0).reshape(B,C*8,D//2,H//2,W//2)
            all_coeffs.append(wt_out)
        return torch.cat(all_coeffs, dim=1)

# ---- DoubleConv (原版完全一致) ----
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv3d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.InstanceNorm3d(out_channels), nn.LeakyReLU(0.01, inplace=True),
            nn.Conv3d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.InstanceNorm3d(out_channels), nn.LeakyReLU(0.01, inplace=True))
        self.residual = nn.Conv3d(in_channels,out_channels,1) \
                        if in_channels!=out_channels else nn.Identity()
    def forward(self, x): return self.conv(x)+self.residual(x)

print('✅ MultiScaleWavelet3D, DoubleConv')

✅ MultiScaleWavelet3D, DoubleConv


In [4]:
# ---- FrequencyBranchV4 (Full, 原版完全一致) ----
# 注意: band_energies 只取 band_features[:8]，shape=(B,8)
# 對應 AdaptiveRouter 的輸入 channels*2 + 8
class FrequencyBranchV4(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.wavelet_scales = ['db1','db2','db4']
        self.multiscale_wavelet = MultiScaleWavelet3D(self.wavelet_scales)
        total_bands = 24
        self.band_weights = nn.Parameter(torch.ones(total_bands)/total_bands)
        self.band_convs = nn.ModuleList([
            nn.Sequential(nn.Conv3d(channels,channels//4,1,bias=False),
                          nn.InstanceNorm3d(channels//4), nn.LeakyReLU(0.01))
            for _ in range(total_bands)])
        self.process = nn.Sequential(
            self._dconv(channels*6, channels*4),
            self._dconv(channels*4, channels*2),
            self._dconv(channels*2, channels))
    def _dconv(self, inc, outc):
        return nn.Sequential(
            nn.Conv3d(inc,outc,3,padding=1,bias=False), nn.InstanceNorm3d(outc), nn.LeakyReLU(0.01,inplace=True),
            nn.Conv3d(outc,outc,3,padding=1,bias=False), nn.InstanceNorm3d(outc), nn.LeakyReLU(0.01,inplace=True))
    def forward(self, x):
        B,C,D,H,W = x.shape
        wc = self.multiscale_wavelet(x)
        band_features = []
        for i in range(len(self.band_weights)):
            band = wc[:, i*C:(i+1)*C]
            feat = self.band_convs[i](band) * self.band_weights[i]
            feat = F.interpolate(feat, size=(D,H,W), mode='trilinear', align_corners=False)
            band_features.append(feat)
        merged = torch.cat(band_features, dim=1)
        output = self.process(merged)
        # ★ 只取前 8 個 band → shape (B, 8)
        band_energies = torch.stack(
            [torch.mean(torch.abs(f), dim=[2,3,4]) for f in band_features[:8]], dim=1)
        return output, band_energies

print('✅ FrequencyBranchV4 (Full)')

✅ FrequencyBranchV4 (Full)


In [5]:
# ---- FrequencyBranchLite (新增，Depthwise Separable) ----
# band_energies 同樣只取 [:8]，保持與 AdaptiveRouter 介面一致
class FrequencyBranchLite(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.wavelet_scales = ['db1','db2','db4']
        self.multiscale_wavelet = MultiScaleWavelet3D(self.wavelet_scales)
        total_bands = 24
        self.band_weights = nn.Parameter(torch.ones(total_bands)/total_bands)
        self.band_convs = nn.ModuleList([
            nn.Sequential(nn.Conv3d(channels,channels//4,1,bias=False),
                          nn.InstanceNorm3d(channels//4), nn.LeakyReLU(0.01))
            for _ in range(total_bands)])
        # ★ 核心差異: pointwise + depthwise separable 替換重型 fusion
        self.pw_reduce = nn.Sequential(
            nn.Conv3d(channels*6, channels, 1, bias=False),
            nn.InstanceNorm3d(channels), nn.LeakyReLU(0.01, inplace=True))
        self.dw1 = nn.Sequential(
            nn.Conv3d(channels,channels,3,padding=1,groups=channels,bias=False),
            nn.InstanceNorm3d(channels), nn.LeakyReLU(0.01,inplace=True),
            nn.Conv3d(channels,channels,1,bias=False),
            nn.InstanceNorm3d(channels), nn.LeakyReLU(0.01,inplace=True))
        self.dw2 = nn.Sequential(
            nn.Conv3d(channels,channels,3,padding=1,groups=channels,bias=False),
            nn.InstanceNorm3d(channels), nn.LeakyReLU(0.01,inplace=True),
            nn.Conv3d(channels,channels,1,bias=False),
            nn.InstanceNorm3d(channels), nn.LeakyReLU(0.01,inplace=True))
    def forward(self, x):
        B,C,D,H,W = x.shape
        wc = self.multiscale_wavelet(x)
        band_features = []
        for i in range(len(self.band_weights)):
            band = wc[:, i*C:(i+1)*C]
            feat = self.band_convs[i](band) * self.band_weights[i]
            feat = F.interpolate(feat, size=(D,H,W), mode='trilinear', align_corners=False)
            band_features.append(feat)
        merged = torch.cat(band_features, dim=1)
        out = self.pw_reduce(merged)
        out = self.dw1(out) + out
        out = self.dw2(out) + out
        band_energies = torch.stack(
            [torch.mean(torch.abs(f), dim=[2,3,4]) for f in band_features[:8]], dim=1)
        return out, band_energies

print('✅ FrequencyBranchLite')

✅ FrequencyBranchLite


In [13]:
# ---- CrossDomainAttention & AdaptiveRouter (原版完全一致) ----
class CrossDomainAttention(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.q_spatial=nn.Conv3d(channels,channels//8,1); self.k_freq=nn.Conv3d(channels,channels//8,1)
        self.v_freq=nn.Conv3d(channels,channels,1); self.q_freq=nn.Conv3d(channels,channels//8,1)
        self.k_spatial=nn.Conv3d(channels,channels//8,1); self.v_spatial=nn.Conv3d(channels,channels,1)
        self.gamma_s=nn.Parameter(torch.zeros(1)); self.gamma_f=nn.Parameter(torch.zeros(1))
    def forward(self, spatial, freq):
        B,C,D,H,W = spatial.shape
        qs=self.q_spatial(spatial).view(B,-1,D*H*W); kf=self.k_freq(freq).view(B,-1,D*H*W)
        vf=self.v_freq(freq).view(B,-1,D*H*W)
        attn_s=F.softmax(torch.bmm(qs.transpose(1,2),kf),dim=-1)
        out_s=torch.bmm(vf,attn_s.transpose(1,2)).view(B,C,D,H,W)
        spatial_refined=spatial+self.gamma_s*out_s
        qf=self.q_freq(freq).view(B,-1,D*H*W); ks=self.k_spatial(spatial).view(B,-1,D*H*W)
        vs=self.v_spatial(spatial).view(B,-1,D*H*W)
        attn_f=F.softmax(torch.bmm(qf.transpose(1,2),ks),dim=-1)
        out_f=torch.bmm(vs,attn_f.transpose(1,2)).view(B,C,D,H,W)
        return spatial_refined, freq+self.gamma_f*out_f

class AdaptiveRouter(nn.Module):
    def __init__(self, channels=512):
        super().__init__()
        # channels*2 spatial stats + 8 band energies
        self.fc = nn.Sequential(
            nn.Linear(channels*2+8,128), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(128,2), nn.Softmax(dim=1))
    def forward(self, spatial, freq, band_energies):
        stats = torch.cat([spatial.mean(dim=[2,3,4]), spatial.std(dim=[2,3,4])], dim=1)
        be = band_energies.view(band_energies.shape[0], -1)[:, :8]  # 確保 shape=(B,8)
        combined = torch.cat([stats, be], dim=1)
        return self.fc(combined)

print('✅ CrossDomainAttention, AdaptiveRouter')

✅ CrossDomainAttention, AdaptiveRouter


In [14]:
# ---- AFS_DSN_V4 & AFS_DSN_Lite (共用 forward，只換 FrequencyBranch) ----
class _AFS_DSN_Base(nn.Module):
    def __init__(self, in_ch, num_classes, base_features,
                 use_freq_branch, use_cross_attention, use_router, freq_cls):
        super().__init__()
        f=base_features
        self.use_freq_branch=use_freq_branch
        self.use_cross_attention=use_cross_attention and use_freq_branch
        self.use_router=use_router and use_freq_branch
        self.encoder1=DoubleConv(in_ch,f); self.encoder2=DoubleConv(f,f*2)
        self.encoder3=DoubleConv(f*2,f*4); self.encoder4=DoubleConv(f*4,f*8)
        self.pool=nn.MaxPool3d(2); self.bottleneck=DoubleConv(f*8,f*16)
        if self.use_freq_branch: self.freq_branch=freq_cls(f*16)
        if self.use_cross_attention: self.cross_attention=CrossDomainAttention(f*16)
        if self.use_router: self.router=AdaptiveRouter(channels=f*16)
        self.up4=nn.ConvTranspose3d(f*16,f*8,2,stride=2); self.decoder4=DoubleConv(f*16,f*8)
        self.up3=nn.ConvTranspose3d(f*8,f*4,2,stride=2);  self.decoder3=DoubleConv(f*8,f*4)
        self.up2=nn.ConvTranspose3d(f*4,f*2,2,stride=2);  self.decoder2=DoubleConv(f*4,f*2)
        self.up1=nn.ConvTranspose3d(f*2,f,2,stride=2);    self.decoder1=DoubleConv(f*2,f)
        self.final=nn.Conv3d(f,num_classes,1)
    def forward(self, x):
        e1=self.encoder1(x); e2=self.encoder2(self.pool(e1))
        e3=self.encoder3(self.pool(e2)); e4=self.encoder4(self.pool(e3))
        b=self.bottleneck(self.pool(e4))
        routing_weights=band_energies=None
        if self.use_freq_branch:
            freq_feat,band_energies=self.freq_branch(b)
            if self.use_cross_attention:
                spatial_refined,freq_refined=self.cross_attention(b,freq_feat)
            else:
                spatial_refined,freq_refined=b,freq_feat
            if self.use_router:
                routing_weights=self.router(spatial_refined,freq_refined,band_energies)
                w_s=routing_weights[:,0:1,None,None,None]; w_f=routing_weights[:,1:2,None,None,None]
                b=spatial_refined*w_s+freq_refined*w_f
            else:
                b=(spatial_refined+freq_refined)/2
        d4=self.decoder4(torch.cat([self.up4(b),e4],dim=1))
        d3=self.decoder3(torch.cat([self.up3(d4),e3],dim=1))
        d2=self.decoder2(torch.cat([self.up2(d3),e2],dim=1))
        d1=self.decoder1(torch.cat([self.up1(d2),e1],dim=1))
        return {'output':self.final(d1),'routing_weights':routing_weights,'band_energies':band_energies}

class AFS_DSN_V4(_AFS_DSN_Base):
    """論文原版 Full model (~414M params)"""
    def __init__(self,in_channels=1,num_classes=2,base_features=32,
                 use_freq_branch=True,use_cross_attention=True,use_router=True):
        super().__init__(in_channels,num_classes,base_features,
                         use_freq_branch,use_cross_attention,use_router,FrequencyBranchV4)

class AFS_DSN_Lite(_AFS_DSN_Base):
    """Lite model (~80M params)"""
    def __init__(self,in_channels=1,num_classes=2,base_features=32,
                 use_freq_branch=True,use_cross_attention=True,use_router=True):
        super().__init__(in_channels,num_classes,base_features,
                         use_freq_branch,use_cross_attention,use_router,FrequencyBranchLite)

def count_params(model): return sum(p.numel() for p in model.parameters())

for cls,name in [(AFS_DSN_V4,'V4 Full'),(AFS_DSN_Lite,'Lite')]:
    m=cls().to(device)
    print(f'AFS_DSN_{name}: {count_params(m)/1e6:.2f}M params')
    del m
torch.cuda.empty_cache()
print('✅ AFS_DSN_V4, AFS_DSN_Lite')

AFS_DSN_V4 Full: 414.57M params
AFS_DSN_Lite: 27.41M params
✅ AFS_DSN_V4, AFS_DSN_Lite


In [15]:
# ---- NasalSegDataset (原版完全一致，無增強) ----
class NasalSegDataset(Dataset):
    def __init__(self, data_root, target_size=(128,128,128)):
        self.data_root=Path(data_root); self.target_size=target_size
        for img_dir,lbl_dir in [
            (Path(data_root)/'NasalSeg'/'images', Path(data_root)/'NasalSeg'/'labels'),
            (Path(data_root)/'images',             Path(data_root)/'labels')]:
            if img_dir.exists() and lbl_dir.exists():
                self.images_dir,self.labels_dir=img_dir,lbl_dir; break
        else: raise FileNotFoundError(f'Cannot find images/labels under {data_root}')
        self.samples=[]
        for img_file in sorted(self.images_dir.glob('*_img.nrrd')):
            pid=img_file.stem.replace('_img','')
            seg=self.labels_dir/f'{pid}_seg.nrrd'
            if seg.exists(): self.samples.append((img_file,seg))
        print(f'📊 Found {len(self.samples)} samples')
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        img_path,seg_path=self.samples[idx]
        image,_=nrrd.read(str(img_path)); mask,_=nrrd.read(str(seg_path))
        image=(image-image.mean())/(image.std()+1e-8)
        image_t=torch.from_numpy(image).float().unsqueeze(0).unsqueeze(0)
        mask_t=torch.from_numpy(mask).long().unsqueeze(0).unsqueeze(0)
        image_t=F.interpolate(image_t,size=self.target_size,mode='trilinear',align_corners=False)
        mask_t=F.interpolate(mask_t.float(),size=self.target_size,mode='nearest').long()
        image_t=image_t.squeeze(0); mask_t=(mask_t.squeeze(0).squeeze(0)>0).long()
        return image_t, mask_t

print('✅ NasalSegDataset')

✅ NasalSegDataset


In [16]:
# ---- Loss & metrics (原版完全一致) ----
class CombinedLoss(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.ce = nn.CrossEntropyLoss()
        self.num_classes = num_classes

    def dice_loss(self, pred, target):
        smooth = 1e-5
        pred = F.softmax(pred, dim=1)
        toh = F.one_hot(target, self.num_classes).permute(0, 4, 1, 2, 3).float()
        inter = (pred * toh).sum(dim=(2, 3, 4))
        union = pred.sum(dim=(2, 3, 4)) + toh.sum(dim=(2, 3, 4))
        dice = (2. * inter + smooth) / (union + smooth)
        return 1 - dice.mean()

    def forward(self, pred, target):
        return self.ce(pred, target) + self.dice_loss(pred, target)

def dice_coefficient(pred, target, num_classes=2):
    """Batch Dice for training monitoring"""
    pred=torch.argmax(pred,dim=1); scores=[]
    for c in range(1,num_classes):
        pc=(pred==c).float(); tc=(target==c).float()
        inter=(pc*tc).sum(); union=pc.sum()+tc.sum()
        scores.append(float((2.*inter)/(union+1e-8)) if union>0 else 1.0)
    return float(np.mean(scores)) if scores else 0.0

def compute_dice_np(pred, target, smooth=1e-6):
    """Numpy Dice for per-case evaluation"""
    inter=(pred*target).sum()
    return float((2*inter+smooth)/(pred.sum()+target.sum()+smooth))

def save_checkpoint(model, optimizer, epoch, best_dice, path, variant):
    torch.save({'epoch':epoch,'model':model.state_dict(),
                'optimizer':optimizer.state_dict(),
                'best_dice':best_dice,'variant':variant}, path)
    print(f'💾 Saved: {path}  (epoch {epoch}, dice {best_dice:.4f})')

print('✅ CombinedLoss, dice_coefficient, compute_dice_np')
print('='*50)
print('02_model.ipynb ready')
print('  AFS_DSN_V4()   — full (~414M)')
print('  AFS_DSN_Lite() — lite (~80M)')
print('  NasalSegDataset(root)')
print('  CombinedLoss() — CE + Dice')
print('='*50)

✅ CombinedLoss, dice_coefficient, compute_dice_np
02_model.ipynb ready
  AFS_DSN_V4()   — full (~414M)
  AFS_DSN_Lite() — lite (~80M)
  NasalSegDataset(root)
  CombinedLoss() — CE + Dice
